# Chapter 15: Eigendecomposition and SVD Applications

**Topics Covered:**
- PCA from Scratch
- Dimensionality Reduction
- Face Recognition (Eigenfaces)
- Recommender System via SVD
- Spectral Clustering


## PCA from Scratch

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
iris = load_iris()
X = StandardScaler().fit_transform(iris.data)
# Covariance → Eigendecomposition
cov = np.cov(X.T)
eigvals, eigvecs = np.linalg.eigh(cov)
# Sort descending
idx = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[idx], eigvecs[:,idx]
print('Explained variance ratio:', np.round(eigvals/eigvals.sum(), 4))
X_pca = X @ eigvecs[:,:2]
plt.figure(figsize=(6,4))
for c, label in zip([0,1,2], iris.target_names):
    mask = iris.target == c
    plt.scatter(X_pca[mask,0], X_pca[mask,1], label=label)
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.legend(); plt.title('PCA - Iris Dataset'); plt.grid(True); plt.show()

## PCA for Compression

In [ ]:
np.random.seed(5)
X = np.random.multivariate_normal([0,0,0], [[4,2,1],[2,3,1],[1,1,2]], 200)
X_c = X - X.mean(axis=0)
U, s, Vt = np.linalg.svd(X_c, full_matrices=False)
for k in [1, 2, 3]:
    X_reduced = X_c @ Vt[:k].T
    X_recon = X_reduced @ Vt[:k]
    err = np.linalg.norm(X_c - X_recon, 'fro')
    var = np.sum(s[:k]**2) / np.sum(s**2)
    print(f'k={k}: var explained={var:.3f}, recon error={err:.3f}')

## Eigenfaces Concept

In [ ]:
from sklearn.datasets import fetch_olivetti_faces
faces = fetch_olivetti_faces(shuffle=True, random_state=42)
X = faces.data  # 400 x 4096
X_c = X - X.mean(axis=0)
U, s, Vt = np.linalg.svd(X_c, full_matrices=False)
# Top 6 eigenfaces
fig, axes = plt.subplots(2, 3, figsize=(9,6))
for i, ax in enumerate(axes.flat):
    ax.imshow(Vt[i].reshape(64,64), cmap='gray')
    ax.set_title(f'Eigenface {i+1}'); ax.axis('off')
plt.suptitle('Top 6 Eigenfaces'); plt.tight_layout(); plt.show()

## SVD-based Recommender System

In [ ]:
# User-item rating matrix (0 = no rating)
R = np.array([[5,3,0,1],[4,0,0,1],[1,1,0,5],[1,0,0,4],[0,1,5,4]], dtype=float)
# Replace 0s with mean for SVD
R_mean = R.copy()
R_mean[R==0] = R[R>0].mean()
U, s, Vt = np.linalg.svd(R_mean, full_matrices=False)
# Low-rank approximation (k=2)
k=2; R_pred = U[:,:k] @ np.diag(s[:k]) @ Vt[:k,:]
print('Predicted ratings:\n', np.round(R_pred, 2))
print('\nUser 2, Item 2 predicted rating:', round(R_pred[1,2], 2))

## Spectral Clustering Preview

In [ ]:
from sklearn.datasets import make_moons
X, y = make_moons(n_samples=100, noise=0.05, random_state=42)
# Similarity matrix (Gaussian kernel)
from scipy.spatial.distance import cdist
dist = cdist(X, X)
sigma = 0.5
W = np.exp(-dist**2 / (2*sigma**2))
# Laplacian
D = np.diag(W.sum(axis=1))
L = D - W
# Smallest non-zero eigenvectors
eigvals, eigvecs = np.linalg.eigh(L)
plt.figure(figsize=(10,4))
plt.subplot(121); plt.scatter(X[:,0],X[:,1],c=y,cmap='bwr'); plt.title('True Labels')
plt.subplot(122); plt.scatter(eigvecs[:,1],eigvecs[:,2],c=y,cmap='bwr'); plt.title('Spectral Embedding')
plt.tight_layout(); plt.show()

## 📝 Summary

Chapter 15 brings together eigendecomposition and SVD to tackle high-impact machine learning applications. PCA uses the eigenvectors of the covariance matrix (or SVD of the centered data matrix) to find the directions of maximum variance, enabling dimensionality reduction and visualization. Eigenfaces demonstrates how SVD can compress and recognize face images by representing them as combinations of principal components. SVD-based collaborative filtering powers recommendation engines by factoring sparse user-item matrices. Spectral clustering uses graph Laplacian eigenvectors to discover non-linear cluster structures — a powerful example of geometry through linear algebra.